# Модуль 03 · Урок 2. Приведение типов, nullable-значения и проверка ошибок

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Почему типы важны
После чтения CSV цены могут оказаться строками: `'100' + '20'` даст `'10020'`, тогда как арифметически ожидается 120. У каждой Series есть `dtype`; проверяйте `df.dtypes`, а не оценивайте тип по внешнему виду. `pd.to_numeric(series, errors='coerce')` превращает некорректные значения в пропуски; при `errors='raise'` ошибка прерывает выполнение и показывает проблему. Выбор зависит от того, что важнее: терпимость к плохим данным или строгая остановка обработки.

In [ ]:
import pandas as pd
values = pd.Series(['14.5', '5', 'n/a', '-3'])
nums = pd.to_numeric(values,errors='coerce')
print(nums, nums.sum())

## 2. Nullable-типы и целые значения
Обычный тип `int64` NumPy не умеет хранить `NaN`, поэтому при пропусках столбец исторически часто становился float. Nullable `Int64` (с заглавной I) позволяет хранить целые числа и `pd.NA`; `boolean` — логические данные с пропуском, `string` — строковый тип pandas. Это полезно для ID, количества единиц и признаков, у которых отсутствующее значение не должно превратиться в ложное.

In [ ]:
items = pd.Series(['2',None,'5'])
quantities = pd.to_numeric(items,errors='coerce').astype('Int64')
print(quantities, quantities.dtype)

## 3. Текст перед типизацией
Пробелы, запятые как десятичный разделитель и единицы измерения требуют явного преобразования. Например, `str.replace(',', '.', regex=False)` до `to_numeric`; предварительно решите, допустимы ли и другие символы. Не удаляйте произвольно все буквы из суммы: `'12руб'` можно привести по правилу, но `'неизвестно'` должно остаться подозрительным и попасть в отчёт.

In [ ]:
weights = pd.Series([' 1,5 ','3,0','не задано'])
normalized = weights.str.strip().str.replace(',','.',regex=False)
print(pd.to_numeric(normalized,errors='coerce'))

## 4. Семантические проверки
Корректный числовой тип не делает значение валидным. Цена может быть отрицательной, скидка — больше 100%, а количество — дробным. Поэтому отдельно проверяйте диапазоны с `between()`, допустимость категорий и уникальность идентификаторов. Выходные данные удобнее хранить отдельно от отчёта о проблемах.

**Проверьте себя:** чем `errors='coerce'` отличается от `raise`? Почему `Int64` и `int64` не одно и то же? Может ли отрицательная цена быть числом, но невалидным значением?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m3_06 · parse price

Создайте `price_num` из текстовой `price` через числовое преобразование: ошибки и пустые строки станут пропусками. Остальные поля не меняйте.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_06: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_06 = expected_for("m3_06", df=df)


In [ ]:
# m3_06: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 6 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`, `price_num`.

```text
     item   price quantity category          email  price_num
11   A-1   128.75        2    FOOD   A@EXAMPLE.COM     128.75
2     b-2                3     food        b@x.ru         NaN
3    A-1   120.50        2    FOOD   A@EXAMPLE.COM     120.50
8     C-3  ошибка     None      TOY            bad        NaN
10   None       0        7     None                      0.00
15   d-4    -4.00        1      toy           None      -4.00
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_06", result, _expected_m3_06)

### m3_07 · parse quantity

Преобразуйте `quantity` в nullable-целочисленный тип `Int64` без изменения других столбцов.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_07: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_07 = expected_for("m3_07", df=df)


In [ ]:
# m3_07: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 5 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`.

```text
     item   price  quantity category          email
11   A-1   128.75         2    FOOD   A@EXAMPLE.COM
2     b-2                 3     food        b@x.ru 
3    A-1   120.50         2    FOOD   A@EXAMPLE.COM
8     C-3  ошибка      <NA>      TOY            bad
10   None       0         7     None               
15   d-4    -4.00         1      toy           None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_07", result, _expected_m3_07)

### m3_08 · normalize decimal

В Series с текстом возможна десятичная запятая и пробелы; получите числовую Series, некорректные значения — NaN.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `values` — `pd.Series` (индекс и значения определены в ячейке примера).


**Выполнение:** используйте готовые переменные `values`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_08: используются переменные values

values = pd.Series([' 12,5', '3.25', 'нет', None], index=[0, 1, 2, 3])

print("values — демонстрационные данные:")

display(values)


# Снимок правильного результата для автопроверки.
_expected_m3_08 = expected_for("m3_08", values=values)


In [ ]:
# m3_08: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: values
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 4 значений; имя серии: `None`.

```text
0    12.50
1     3.25
2      NaN
3      NaN
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_08", result, _expected_m3_08)

### m3_09 · invalid prices

Получите таблицу только с теми строками, где цена после числового разбора не является неотрицательным числом (включая пропуски).

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_09: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_09 = expected_for("m3_09", df=df)


In [ ]:
# m3_09: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 5 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`.

```text
     item   price quantity category     email
2     b-2                3     food   b@x.ru 
8     C-3  ошибка     None      TOY       bad
15   d-4    -4.00        1      toy      None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_09", result, _expected_m3_09)

### m3_10 · safe total

Добавьте `total`: очищенная числовая цена × очищенное количество; при ошибке в любом поле результат пропуск.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_10: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_10 = expected_for("m3_10", df=df)


In [ ]:
# m3_10: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 6 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`, `total`.

```text
     item   price quantity category          email  total
11   A-1   128.75        2    FOOD   A@EXAMPLE.COM  257.5
2     b-2                3     food        b@x.ru     NaN
3    A-1   120.50        2    FOOD   A@EXAMPLE.COM  241.0
8     C-3  ошибка     None      TOY            bad    NaN
10   None       0        7     None                   0.0
15   d-4    -4.00        1      toy           None   -4.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_10", result, _expected_m3_10)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()